# Costo de Mantenimiento de Flota — Validación Temporal y Diagnóstico
## Área: Logística · Transportes del Pacífico S.A. de C.V.

---

## Contexto

**Empresa:** Transportes del Pacífico, 450 unidades de carga en rutas nacionales.

**Problema:** el área de finanzas usa el promedio histórico de los últimos 3 años como
presupuesto de mantenimiento. El director de operaciones sospecha que el modelo puede
ser más preciso — y que el presupuesto actual podría estar subestimando los costos reales.

**Tu tarea:** construir un modelo de predicción de costo mensual de mantenimiento por
unidad, validarlo correctamente con datos históricos de 2019–2024, y determinar si el
modelo sigue siendo válido o necesita actualización.

---

## Datos disponibles

Los datos cubren 2019–2024. Cada registro es el costo mensual de mantenimiento
de una unidad, con las variables operativas de ese mes.

**Fuentes:**
- Inflación de refacciones: INEGI INPC, subíndice 1.1.4 "Refacciones y accesorios"
  https://www.inegi.org.mx/temas/inpc/ — ~8% anual en México 2019-2024.
- `costo_2019_k` = costo deflactado a pesos de 2019 con ese índice INPC.

---

## Por qué usamos holdout temporal fijo para comparar modelos

Comparar modelos con datos temporales requiere respetar el orden del tiempo:
**siempre entrenar con el pasado, validar con el futuro inmediato.**

K-Fold estándar mezcla años y está descartado. Usamos un holdout fijo:
- **Train:** 2019–2020
- **Validación:** 2021 (el año inmediatamente siguiente — el primero "fuera de muestra")

`random_state = 2024`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings; warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

np.random.seed(2024)
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})

INF_REF = 0.08

def idx_inpc(anio, base=2019):
    return (1 + INF_REF)**(anio - base)

print("Índice INPC refacciones (estimado, base 2019=1.0):")
for a in range(2019, 2025):
    print(f"  {a}: {idx_inpc(a):.4f}  (+{(idx_inpc(a)-1)*100:.0f}% vs 2019)")
print()
print("Fuente: INEGI INPC subíndice 1.1.4 — https://www.inegi.org.mx/temas/inpc/")

---
## Sección 1 — Portafolio de datos (2019–2024)

In [ ]:
# ── Portafolio Transportes del Pacífico 2019–2024 — NO MODIFICAR ─────────────
np.random.seed(2024)
ANIOS  = list(range(2019, 2025))
N_ANIO = {2019:800, 2020:750, 2021:850, 2022:950, 2023:1000, 2024:1050}

def _costo_por_ton(anio):
    return 180 + 60*max(0, anio - 2021)

def _carga_media(anio):
    return 14 + 3*max(0, anio - 2021)

def p_tipo_ruta(anio):
    urbana    = max(0.14, 0.30 - 0.04*max(0, anio-2021))
    carretera = min(0.57, 0.45 + 0.03*max(0, anio-2021))
    mixta     = 1 - urbana - carretera
    return [urbana, carretera, mixta]

registros = []
for anio in ANIOS:
    n    = N_ANIO[anio]
    km   = np.clip(np.random.normal(8500, 2200, n), 1000, 18000).round(0)
    an_u = np.random.randint(0, 16, n)
    tipo = np.random.choice([0,1,2], n, p=p_tipo_ruta(anio))
    tm   = _carga_media(anio)
    ton  = np.clip(np.random.normal(tm, 5, n), 2, 34).round(1)
    marc = np.random.choice([0,1,2], n, p=[.45,.35,.20])
    km_u = np.clip(np.random.normal(3500, 1200, n), 500, 8000).round(0)
    ct   = _costo_por_ton(anio)

    cb = (1500 + 0.18*km + 120*an_u + 500*(tipo==0)
          + ct*ton + 300*(marc==1) + 800*(marc==2)
          + 0.12*km_u + np.abs(np.random.normal(0, 400, n)))
    cn = (cb * idx_inpc(anio)).round(2)

    for i in range(n):
        registros.append({
            'anio': anio, 'km_recorridos': round(km[i]),
            'anios_unidad': int(an_u[i]), 'tipo_ruta': int(tipo[i]),
            'toneladas_cargadas': round(ton[i], 1), 'marca': int(marc[i]),
            'km_desde_ultimo_mto': round(km_u[i]),
            'costo_nominal_k': round(cn[i]/1000, 2),
            'costo_2019_k':    round(cb[i]/1000, 2),
            'idx': idx_inpc(anio)
        })

df = pd.DataFrame(registros).sort_values('anio').reset_index(drop=True)
FEATURES = ['km_recorridos','anios_unidad','tipo_ruta',
            'toneladas_cargadas','marca','km_desde_ultimo_mto']

print(f"Transportes del Pacífico: {len(df):,} registros de mantenimiento 2019–2024")
print()
print(f"  {'Año':>5}  {'n':>5}  {'Costo nominal':>15}  {'Costo deflact.':>15}  {'Índice INPC':>12}")
for a in ANIOS:
    s = df[df.anio==a]
    print(f"  {a:>5}  {len(s):>5,}  ${s.costo_nominal_k.mean():>13.3f}K  "
          f"${s.costo_2019_k.mean():>13.3f}K  {s.idx.mean():>12.4f}")
print()
print("OBSERVA: el costo nominal sube cada año.")
print("¿Es inflación? ¿Es algo más? El análisis siguiente lo determinará.")

---
## Sección 2 — Comparación temporal de modelos (sin K-Fold estándar)

In [ ]:
# ── Holdout temporal fijo: train 2019-2020, validación 2021 ─────────────────
mask_tr = df['anio'].isin([2019, 2020])
mask_va = df['anio'] == 2021

X_tr = df.loc[mask_tr, FEATURES].values;  y_tr = df.loc[mask_tr, 'costo_nominal_k'].values
X_va = df.loc[mask_va, FEATURES].values;  y_va = df.loc[mask_va, 'costo_nominal_k'].values

modelos = {
    'M1 — Ridge (lineal)':           Pipeline([('sc',StandardScaler()),('m',Ridge(alpha=1))]),
    'M2 — Árbol de decisión':        Pipeline([('sc',StandardScaler()),
                                               ('m',DecisionTreeRegressor(max_depth=6,random_state=2024))]),
    'M3 — GBR (Gradient Boosting)':  Pipeline([('sc',StandardScaler()),
                                               ('m',GradientBoostingRegressor(n_estimators=150,
                                                    max_depth=4, learning_rate=0.08,
                                                    random_state=2024))]),
}

print("COMPARACIÓN — Holdout temporal fijo")
print(f"  Train: 2019–2020 ({mask_tr.sum():,} registros)  "
      f"Val: 2021 ({mask_va.sum():,} registros)")
print()
print(f"  {'Modelo':40s}  {'MAE ($K)':>10}  {'RMSE ($K)':>11}  {'R²':>8}  {'MAPE%':>7}")
print("-"*83)

holdout_results = {}
for nombre, pipe in modelos.items():
    pipe.fit(X_tr, y_tr)
    yp   = pipe.predict(X_va)
    mae  = mean_absolute_error(y_va, yp)
    rmse = np.sqrt(mean_squared_error(y_va, yp))
    r2   = r2_score(y_va, yp)
    mape = np.mean(np.abs((y_va - yp) / y_va)) * 100
    holdout_results[nombre] = {'mae':mae,'rmse':rmse,'r2':r2,'mape':mape}
    print(f"  {nombre:40s}  {mae:>9.3f}  {rmse:>10.3f}  {r2:>8.4f}  {mape:>6.1f}%")

---
## Sección 3 — Selección del modelo

| Criterio | Ridge | Árbol | **GBR** |
|----------|-------|-------|---------|
| MAE holdout 2021 | ~$0.95K | ~$1.00K | **~$0.95K** |
| MAPE | 10.8% | 11.2% | **10.6%** |
| R² | 0.46 | 0.26 | 0.45 |
| Captura interacciones no lineales | ❌ | ✅ pero sobreajusta | **✅** |
| Riesgo de sobreajuste | Bajo | Alto | Bajo |

Ridge y GBR tienen MAE casi idéntico en 2021. La diferencia está en la estructura:
Ridge solo puede capturar relaciones lineales entre variables. GBR captura interacciones
(por ejemplo, el efecto de toneladas varía según el tipo de ruta).

El árbol muestra R²=0.26 — señal de sobreajuste (aprende el ruido del train).

**GBR seleccionado** por menor riesgo de sobreajuste y capacidad de capturar
relaciones no lineales que pueden volverse relevantes en datos futuros.

In [ ]:
# ── Visualización comparación de modelos ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Selección de modelo — Transportes del Pacífico (holdout 2021)',
             fontweight='bold')

etiquetas = ['Ridge', 'Árbol', 'GBR']
colores   = ['#94A3B8','#F59E0B','#059669']
nombres_l = list(modelos.keys())
maes_h = [holdout_results[n]['mae']  for n in nombres_l]
r2s_h  = [holdout_results[n]['r2']   for n in nombres_l]
mapes  = [holdout_results[n]['mape'] for n in nombres_l]
rmses  = [holdout_results[n]['rmse'] for n in nombres_l]

# Panel 1: MAE
ax = axes[0]
ax.bar(etiquetas, maes_h, color=colores, alpha=0.85, edgecolor='white')
for i,(e,v) in enumerate(zip(etiquetas, maes_h)):
    ax.text(i, v+0.005, f'${v:.3f}K', ha='center', fontsize=10,
            fontweight='bold', color=colores[i])
ax.set_title('MAE ($K) — Holdout 2021\nMenor = mejor', fontweight='bold')
ax.set_ylabel('MAE ($K MXN)'); ax.grid(True, alpha=0.3, axis='y')

# Panel 2: MAPE
ax = axes[1]
ax.bar(etiquetas, mapes, color=colores, alpha=0.85, edgecolor='white')
for i,(e,v) in enumerate(zip(etiquetas, mapes)):
    ax.text(i, v+0.05, f'{v:.1f}%', ha='center', fontsize=10,
            fontweight='bold', color=colores[i])
ax.set_title('MAPE (%) — Error relativo\nInterpretable para el negocio', fontweight='bold')
ax.set_ylabel('MAPE (%)'); ax.grid(True, alpha=0.3, axis='y')

# Panel 3: Ratio RMSE/MAE
ax = axes[2]
ratios = [holdout_results[n]['rmse']/holdout_results[n]['mae'] for n in nombres_l]
ax.bar(etiquetas, ratios, color=colores, alpha=0.85, edgecolor='white')
for i,(e,v) in enumerate(zip(etiquetas, ratios)):
    ax.text(i, v+0.003, f'{v:.3f}', ha='center', fontsize=10,
            fontweight='bold', color=colores[i])
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.set_title('Ratio RMSE/MAE\n~1.0 = errores uniformes', fontweight='bold')
ax.set_ylabel('RMSE / MAE'); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('flota_seleccion.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 flota_seleccion.png")

---
## Sección 4 — Walk-Forward: ¿inflación, drift, o ambos?

El GBR fue el modelo seleccionado. Ahora lo evaluamos en los años que no vio durante
la selección: 2022, 2023 y 2024.

El Walk-Forward re-entrena con todo el histórico disponible hasta el año de test.
Para cada año de test entrenamos **dos versiones del GBR** — misma arquitectura,
diferente target:
- **Nominal:** target = costo real en pesos del año → captura inflación + cualquier cambio estructural
- **Deflactado:** target = costo en pesos de 2019 → elimina la inflación; lo que queda es solo cambio estructural

Cuando el MAE Walk-Forward sube hay dos explicaciones posibles:
1. **Inflación** — el modelo sigue siendo correcto en términos reales; deflactar el target resuelve
2. **Drift** — las relaciones entre variables y costo cambiaron; hay que re-entrenar

Los datos te dirán cuál de los dos está ocurriendo.

In [ ]:
# ── Walk-Forward: nominal vs deflactado ───────────────────────────────────────
ANIOS_TEST   = [2022, 2023, 2024]
res_nom, res_def = [], []

mae_def_base = None

print("WALK-FORWARD — GBR (re-entrenando con todo el histórico disponible)")
print()
print(f"  {'Año':>5}  {'Train con':>14}  {'MAE Nominal':>13}  {'MAPE Nom%':>11}  "
      f"{'MAE Deflact.':>14}  {'R² Def':>8}")
print("-"*78)

for anio_test in ANIOS_TEST:
    mtr = df['anio'] < anio_test;  mte = df['anio'] == anio_test
    X_tr = df.loc[mtr, FEATURES].values;   X_te = df.loc[mte, FEATURES].values
    anios_tr = sorted(df.loc[mtr,'anio'].unique())
    rango_tr = f"{anios_tr[0]}–{anios_tr[-1]}"

    # Walk-Forward NOMINAL
    pn = Pipeline([('sc',StandardScaler()),
                   ('m',GradientBoostingRegressor(n_estimators=150,max_depth=4,
                                                   learning_rate=0.08,random_state=2024))])
    pn.fit(X_tr, df.loc[mtr,'costo_nominal_k'].values)
    pred_nom = pn.predict(X_te); y_te_nom = df.loc[mte,'costo_nominal_k'].values
    mae_nom  = mean_absolute_error(y_te_nom, pred_nom)
    mape_nom = np.mean(np.abs((y_te_nom - pred_nom) / y_te_nom)) * 100

    # Walk-Forward DEFLACTADO (target en pesos de 2019)
    pd_ = Pipeline([('sc',StandardScaler()),
                    ('m',GradientBoostingRegressor(n_estimators=150,max_depth=4,
                                                    learning_rate=0.08,random_state=2024))])
    pd_.fit(X_tr, df.loc[mtr,'costo_2019_k'].values)
    pred_def = pd_.predict(X_te); y_te_def = df.loc[mte,'costo_2019_k'].values
    mae_def  = mean_absolute_error(y_te_def, pred_def)
    r2_def   = r2_score(y_te_def, pred_def)

    if mae_def_base is None:
        mae_def_base = mae_def

    res_nom.append({'anio':anio_test,'mae_nom':mae_nom,'mape_nom':mape_nom})
    res_def.append({'anio':anio_test,'mae_def':mae_def,'r2_def':r2_def,
                    'pred_def':pred_def,'y_te_nom':y_te_nom,'y_te_def':y_te_def})

    print(f"  {anio_test:>5}  {rango_tr:>14}  ${mae_nom:>11.3f}K  {mape_nom:>10.1f}%  "
          f"${mae_def:>12.3f}K  {r2_def:>8.4f}")

print()
mae_n_v = [r['mae_nom'] for r in res_nom]
mae_d_v = [r['mae_def'] for r in res_def]
print(f"MAE nominal:    ${mae_n_v[0]:.3f}K (2022) → ${mae_n_v[-1]:.3f}K (2024)")
print(f"MAE deflactado: ${mae_d_v[0]:.3f}K (2022) → ${mae_d_v[-1]:.3f}K (2024)")
print()
print("Nota: el R² deflactado puede volverse negativo — significa que el modelo")
print("predice peor que usar la media del período. Es una señal fuerte de cambio")
print("en las relaciones que el modelo aprendió.")
print()
print("¿Qué observas? ¿El MAE deflactado se mantiene estable o también sube?")
print("Esa pregunta define el diagnóstico y la acción a tomar.")

---
## Sección 5 — PSI: ¿cambió la distribución del portafolio?

El MAE puede subir incluso cuando el modelo es bueno, si el portafolio de producción
tiene una distribución de variables muy diferente a la del período de entrenamiento.

El **PSI (Population Stability Index)** mide ese cambio distribución a distribución:
- PSI < 0.10 → estable, el portafolio no cambió
- PSI 0.10–0.25 → cambio moderado, monitorear
- PSI > 0.25 → cambio significativo, investigar

Calculamos el PSI de cada variable comparando la base de entrenamiento (2019–2021)
con cada año de producción (2022, 2023, 2024).

In [ ]:
# ── PSI — Population Stability Index ─────────────────────────────────────────
def calcular_psi(base_values, actual_values, n_bins=10):
    percentiles = np.linspace(0, 100, n_bins+1)
    bins        = np.percentile(base_values, percentiles)
    bins[0]    -= 1e-6;  bins[-1] += 1e-6
    bc = np.histogram(base_values,   bins=bins)[0]
    ac = np.histogram(actual_values, bins=bins)[0]
    bp = np.where(bc==0, 0.001, bc/len(base_values))
    ap = np.where(ac==0, 0.001, ac/len(actual_values))
    return np.sum((ap - bp) * np.log(ap / bp))

base_mask = df['anio'].isin([2019, 2020, 2021])

print("PSI por variable — base: 2019–2021 vs cada año de producción:")
print("  < 0.10: estable  |  0.10–0.25: monitorear  |  > 0.25: cambio significativo")
print()
print(f"  {'Variable':>22}  {'2022':>8}  {'2023':>8}  {'2024':>8}  {'Estado'}")
print("-"*68)

psi_resumen = {}
for feat in FEATURES:
    bv    = df.loc[base_mask, feat].values
    psies = [calcular_psi(bv, df.loc[df['anio']==a, feat].values) for a in [2022,2023,2024]]
    psi_resumen[feat] = psies
    max_p = max(psies)
    estado = "🔴 Cambio severo" if max_p>0.50 else ("⚠️ Monitorear" if max_p>0.10 else "✅ Estable")
    print(f"  {feat:>22}  "+"  ".join(f"{p:>8.4f}" for p in psies)+f"  {estado}")

print()
print("¿Qué variable muestra el cambio más grande?")
print("¿Ese cambio es consistente con el aumento del MAE deflactado que viste antes?")

In [ ]:
# ── Visualización diagnóstico ────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, wspace=0.38, hspace=0.45)

años_p    = [r['anio'] for r in res_nom]
mae_nom_v = [r['mae_nom'] for r in res_nom]
mae_def_v = [r['mae_def'] for r in res_def]

# Panel 1: MAE nominal vs deflactado
ax = fig.add_subplot(gs[0,0])
ax.plot(años_p, mae_nom_v, 'o-', color='#DC2626', lw=2.5, ms=9, label='MAE nominal')
ax.plot(años_p, mae_def_v, 's-', color='#D97706', lw=2.5, ms=9, label='MAE deflactado')
for a,v in zip(años_p,mae_nom_v): ax.text(a,v+0.1,f'${v:.2f}K',ha='center',fontsize=8,color='#DC2626',fontweight='bold')
for a,v in zip(años_p,mae_def_v): ax.text(a,v-0.3,f'${v:.2f}K',ha='center',fontsize=8,color='#D97706',fontweight='bold')
ax.set_title('MAE Walk-Forward\nNominal vs deflactado', fontweight='bold')
ax.set_ylabel('MAE ($K)'); ax.set_xlabel('Año test')
ax.set_xticks(años_p); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 2: Costo medio por año nominal vs deflactado
ax = fig.add_subplot(gs[0,1])
cn = df.groupby('anio')['costo_nominal_k'].mean()
cd = df.groupby('anio')['costo_2019_k'].mean()
ax.plot(cn.index, cn.values, 'o-', color='#DC2626', lw=2.5, ms=8, label='Nominal')
ax.plot(cd.index, cd.values, 's-', color='#059669', lw=2.5, ms=8, label='Deflactado (2019)')
ax.fill_between(cn.index, cd.values, cn.values, alpha=0.15, color='#DC2626', label='Componente inflación')
ax.set_title('Costo medio anual por unidad\nNominal vs deflactado', fontweight='bold')
ax.set_ylabel('$K MXN'); ax.set_xlabel('Año')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 3: PSI por variable en 2024
ax = fig.add_subplot(gs[1,0])
feats_psi   = list(psi_resumen.keys())
psi_24      = [psi_resumen[f][2] for f in feats_psi]
colores_psi = ['#DC2626' if v>0.25 else ('#D97706' if v>0.10 else '#059669') for v in psi_24]
ax.barh(feats_psi, psi_24, color=colores_psi, alpha=0.85, edgecolor='white')
ax.axvline(0.10, color='#D97706', lw=2, ls='--', label='0.10 — monitorear')
ax.axvline(0.25, color='#DC2626', lw=2, ls='--', label='0.25 — cambio')
ax.set_title('PSI por variable (2024 vs 2019–2021)', fontweight='bold')
ax.set_xlabel('PSI'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='x')

# Panel 4: Distribución de la variable con mayor PSI
feat_max_psi = max(psi_resumen, key=lambda f: psi_resumen[f][2])
ax = fig.add_subplot(gs[1,1])
base_vals = df.loc[base_mask, feat_max_psi].values
vals_2024 = df.loc[df['anio']==2024, feat_max_psi].values
ax.hist(base_vals, bins=30, alpha=0.6, color='#0D7490', label='2019–2021 (base)', density=True)
ax.hist(vals_2024, bins=30, alpha=0.6, color='#DC2626', label='2024', density=True)
ax.set_title(f'Distribución de {feat_max_psi}\nBase vs 2024', fontweight='bold')
ax.set_xlabel(feat_max_psi); ax.set_ylabel('Densidad')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

fig.suptitle('Diagnóstico — Transportes del Pacífico (2022–2024)',
             fontsize=13, fontweight='bold')
plt.savefig('flota_diagnostico.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 flota_diagnostico.png")
print()
print("Interpreta los 4 paneles:")
print("  Panel 1: ¿Qué le pasa al MAE deflactado? ¿Es estable o sube?")
print("  Panel 2: ¿La curva deflactada es plana o tiene pendiente positiva?")
print("  Panel 3: ¿Qué variable tiene el PSI más alto? ¿Coincide con el Panel 1?")
print("  Panel 4: ¿Qué cambió en la distribución de esa variable?")

---
## Sección 6 — Re-entrenamiento: ¿qué ventana de datos funciona mejor?

Una vez que el Walk-Forward y el PSI dan señales de cambio en el portafolio,
la siguiente pregunta operativa es: **si re-entrenamos, ¿con qué datos?**

Hay dos estrategias posibles:
- **Histórico completo:** agregar cada año nuevo al conjunto de entrenamiento acumulado
- **Ventana reciente:** usar solo los últimos N años, descartando datos anteriores

Para comparar las estrategias de forma justa fijamos el año de test en **2024**
y variamos únicamente lo que entra al entrenamiento.
Así el único factor que cambia entre filas es la ventana de train.

In [ ]:
# ── Re-entrenamiento: test fijo=2024, variar ventana de train ────────────────
mte      = df['anio'] == 2024
X_te     = df.loc[mte, FEATURES].values
y_te_nom = df.loc[mte, 'costo_nominal_k'].values
y_te_def = df.loc[mte, 'costo_2019_k'].values

estrategias = [
    ("Original  (2019–2020)",  [2019, 2020]),
    ("+1 año    (2019–2021)",  [2019, 2020, 2021]),
    ("+2 años   (2019–2022)",  [2019, 2020, 2021, 2022]),
    ("+3 años   (2019–2023)",  [2019, 2020, 2021, 2022, 2023]),
    ("Reciente  (2021–2023)",  [2021, 2022, 2023]),
    ("Reciente  (2022–2023)",  [2022, 2023]),
]

print("RE-ENTRENAMIENTO — test fijo: 2024")
print("¿Cuánto mejora el modelo al incluir datos más recientes en el entrenamiento?")
print()
print(f"  {'Estrategia de train':26s}  {'n_train':>8}  {'MAE Nom':>10}  "
      f"{'MAPE%':>8}  {'MAE Def':>10}  {'R² Def':>8}")
print("-"*80)

resultados_retraining = []
for label, train_years in estrategias:
    mtr    = df['anio'].isin(train_years)
    X_tr_n = df.loc[mtr, FEATURES].values

    pn = Pipeline([('sc', StandardScaler()),
                   ('m',  GradientBoostingRegressor(n_estimators=150, max_depth=4,
                                                     learning_rate=0.08, random_state=2024))])
    pn.fit(X_tr_n, df.loc[mtr, 'costo_nominal_k'].values)
    yn    = pn.predict(X_te)
    mae_n = mean_absolute_error(y_te_nom, yn)
    mape  = np.mean(np.abs((y_te_nom - yn) / y_te_nom)) * 100

    pd_ = Pipeline([('sc', StandardScaler()),
                    ('m',  GradientBoostingRegressor(n_estimators=150, max_depth=4,
                                                      learning_rate=0.08, random_state=2024))])
    pd_.fit(X_tr_n, df.loc[mtr, 'costo_2019_k'].values)
    yd    = pd_.predict(X_te)
    mae_d = mean_absolute_error(y_te_def, yd)
    r2_d  = r2_score(y_te_def, yd)

    resultados_retraining.append({
        'label': label, 'train_years': train_years,
        'n': mtr.sum(), 'mae_n': mae_n, 'mape': mape,
        'mae_d': mae_d, 'r2_d': r2_d
    })
    print(f"  {label:26s}  {mtr.sum():>8,}  ${mae_n:>8.3f}K  "
          f"{mape:>7.1f}%  ${mae_d:>8.3f}K  {r2_d:>8.4f}")

print()
print("Preguntas para analizar:")
print("  1. ¿El MAE deflactado mejora al agregar datos más recientes?")
print("  2. ¿En qué punto el histórico antiguo deja de ayudar y empieza a contaminar?")
print("  3. ¿Cuál es la estrategia con mejor balance entre MAE deflactado y tamaño de muestra?")

In [ ]:
# ── Visualización del experimento de re-entrenamiento ────────────────────────
import matplotlib.cm as cm

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Re-entrenamiento — test fijo 2024: impacto de la ventana de entrenamiento',
             fontweight='bold')

labels_cortos = ['Orig\n2019-20', '2019-21', '2019-22', '2019-23', '2021-23', '2022-23']
maes_n = [r['mae_n'] for r in resultados_retraining]
maes_d = [r['mae_d'] for r in resultados_retraining]
mapes_ = [r['mape']  for r in resultados_retraining]
ns     = [r['n']     for r in resultados_retraining]
colores = [cm.RdYlGn(0.1 + 0.18*i) for i in range(len(labels_cortos))]

# Panel 1: MAE deflactado — la métrica estructural sin efecto de inflación
ax = axes[0]
ax.bar(labels_cortos, maes_d, color=colores, alpha=0.9, edgecolor='white')
for i,v in enumerate(maes_d):
    ax.text(i, v+0.05, f'${v:.3f}K', ha='center', fontsize=8, fontweight='bold')
ax.axhline(maes_d[0], color='#DC2626', lw=1.5, ls='--', alpha=0.7, label='Referencia original')
ax.set_title('MAE Deflactado por estrategia\n(sin efecto de inflación)', fontweight='bold')
ax.set_ylabel('MAE ($K de 2019)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# Panel 2: MAPE nominal — interpretable para el área de operaciones
ax = axes[1]
ax.bar(labels_cortos, mapes_, color=colores, alpha=0.9, edgecolor='white')
for i,v in enumerate(mapes_):
    ax.text(i, v+0.3, f'{v:.1f}%', ha='center', fontsize=8, fontweight='bold')
ax.axhline(mapes_[0], color='#DC2626', lw=1.5, ls='--', alpha=0.7, label='Referencia original')
ax.set_title('MAPE Nominal (%) por estrategia\nError relativo sobre costo real', fontweight='bold')
ax.set_ylabel('MAPE (%)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# Panel 3: n_train vs MAE deflactado — curva de aprendizaje
ax = axes[2]
idx_grow   = [0, 1, 2, 3]   # histórico creciente
idx_recent = [4, 5]          # ventana reciente

ax.plot([ns[i] for i in idx_grow],   [maes_d[i] for i in idx_grow],
        'o-', color='#0D7490', lw=2.5, ms=9, label='Histórico creciente')
ax.plot([ns[i] for i in idx_recent], [maes_d[i] for i in idx_recent],
        's--', color='#059669', lw=2.5, ms=9, label='Ventana reciente')

for i in idx_grow:
    ax.annotate(labels_cortos[i].replace('\n',' '), (ns[i], maes_d[i]),
                textcoords="offset points", xytext=(6, 5), fontsize=7, color='#0D7490')
for i in idx_recent:
    ax.annotate(labels_cortos[i], (ns[i], maes_d[i]),
                textcoords="offset points", xytext=(6, -12), fontsize=7, color='#059669')

ax.set_title('MAE Deflactado vs tamaño de train\nHistórico creciente vs ventana reciente',
             fontweight='bold')
ax.set_xlabel('n registros de entrenamiento')
ax.set_ylabel('MAE Deflactado ($K 2019)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('flota_retraining.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 flota_retraining.png")
print()
print("Panel 3 — la pregunta clave:")
print("  ¿Agregar más datos históricos siempre mejora el modelo?")
print("  Si los datos viejos representan un régimen diferente al actual,")
print("  pueden añadir ruido en lugar de señal.")
print("  La ventana reciente con menos registros pero datos actuales puede superar")
print("  al histórico completo.")

---
## Sección 7 — Diagnóstico y decisión

Con base en los resultados de todas las secciones anteriores, responde:

**¿El problema es inflación, drift, o ambos?**
El Walk-Forward separó los dos fenómenos con el MAE nominal y el MAE deflactado.
Si el MAE deflactado es estable, deflactar el target es suficiente — no hay que re-entrenar.
Si el MAE deflactado también sube, las relaciones que el modelo aprendió ya no aplican.

**¿Qué variable cambió y qué tan severo fue el cambio?**
El PSI cuantificó el cambio en la distribución de cada variable.
Las variables con PSI alto son candidatas a explicar por qué el modelo se degradó.

**¿Qué estrategia de re-entrenamiento conviene?**
El experimento de la Sección 6 muestra si agregar datos recientes recupera el modelo,
y si mantener todo el histórico ayuda o introduce ruido del régimen anterior.

**Conversación con operaciones:**
Las métricas señalan *qué* cambió y *cuándo*. El área de operaciones dirá *por qué*.
¿Hubo algún cambio en contratos, tipo de carga, rutas o flota en el período marcado?